In [1]:
import pandas as pd
import numpy as np
import anndata as ad

In [2]:
pretrain = ad.read_h5ad("/home/kchen/microbiome/gut_microbiome_GPT/datasets/remove_redundant_oct24/pretrain.h5ad")
wgs = ad.read_h5ad("/home/kchen/microbiome/gut_microbiome_GPT/datasets/metagenomics/merged_wgs_with_taxonomy.h5ad")

/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [3]:
print(pretrain)
print(wgs)

AnnData object with n_obs × n_vars = 105482 × 4296
    obs: 'drr', 'study_id', 'location'
    var: 'taxa'
    varm: 'taxonomy'
AnnData object with n_obs × n_vars = 32382 × 4049
    obs: 'study_name'
    varm: 'taxonomy'


In [4]:
pretrain.varm['taxonomy']['Species'] = None
pretrain.varm['taxonomy']

,Domain,Phylum,Class,Order,Family,Genus,Species
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Atopobiaceae,Tractidigestivibacter,None
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Coriobacteriaceae,Collinsella,None
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Adlercreutzia,None
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegalimassilia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Senegalimassilia,None
Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.Holdemanella,Bacteria,Bacillota,Bacilli,Erysipelotrichales,Erysipelotrichaceae,Holdemanella,None
...,...,...,...,...,...,...,...
Bacteria.Bacteroidota.Bacteroidia.Flavobacteriales.Flavobacteriaceae.Flavirhabdus,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Flavobacteriaceae,Flavirhabdus,None
Bacteria.Pseudomonadota.Alphaproteobacteria.Rhodobacterales.Paracoccaceae.Octadecabacter,Bacteria,Pseudomonadota,Alphaproteobacteria,Rhodobacterales,Paracoccaceae,Octadecabacter,None
Bacteria.Pseudomonadota.Alphaproteobacteria.Acetobacterales.Acetobacteraceae.Swingsia,Bacteria,Pseudomonadota,Alphaproteobacteria,Acetobacterales,Acetobacteraceae,Swingsia,None
Bacteria.Bacteroidota.Bacteroidia.Flavobacteriales.Flavobacteriaceae.Aurantiacicella,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Flavobacteriaceae,Aurantiacicella,None


In [5]:
combined = ad.concat([pretrain, wgs], 
                     axis=0,
                     join='outer',
                     )

/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [6]:
# 3) Make taxonomy DataFrames that are POSITION-aligned to each object's var_names
tax_pre = pretrain.varm["taxonomy"].copy()
tax_pre.index = pretrain.var_names  # force correct index

tax_wgs = wgs.varm["taxonomy"].copy()
tax_wgs.index = wgs.var_names       # force correct index

# 4) Align both to combined.var_names and merge
tax_pre = tax_pre.reindex(combined.var_names)
tax_wgs = tax_wgs.reindex(combined.var_names)

combined_taxonomy = tax_pre.combine_first(tax_wgs)

# 5) Final guarantee: exact index match required by varm
combined_taxonomy = combined_taxonomy.reindex(combined.var_names)
combined.varm["taxonomy"] = combined_taxonomy

In [7]:
combined

AnnData object with n_obs × n_vars = 137864 × 8345
    obs: 'drr', 'study_id', 'location', 'study_name'
    varm: 'taxonomy'

In [8]:
wgs

AnnData object with n_obs × n_vars = 32382 × 4049
    obs: 'study_name'
    varm: 'taxonomy'

In [9]:
pretrain

AnnData object with n_obs × n_vars = 105482 × 4296
    obs: 'drr', 'study_id', 'location'
    var: 'taxa'
    varm: 'taxonomy'

In [10]:
# bytes -> str
combined.varm["taxonomy"] = combined.varm["taxonomy"].applymap(lambda x: x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else x)

# fill missing, then force plain python strings (object dtype)
combined.varm["taxonomy"] = combined.varm["taxonomy"].astype(object).applymap(str)
# combined.write_h5ad("/home/kchen/microbiome/gut_microbiome_GPT/datasets/metagenomics/combined_16s_wgs.h5ad")

/tmp/ipykernel_153787/1242177104.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined.varm["taxonomy"] = combined.varm["taxonomy"].applymap(lambda x: x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else x)
/tmp/ipykernel_153787/1242177104.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined.varm["taxonomy"] = combined.varm["taxonomy"].astype(object).applymap(str)
